In [1]:
import os
if not os.path.exists("/content/model"):
  !git clone https://github.com/ggarciabaez/VLA.git
  %cd VLA
  !git checkout qformer
  %mv model ../
  %mv scripts/train_datasets.py ../
  %mv scripts/train_tools.py ../
  %cd ../
  %rm -r VLA
  !pip install --upgrade sentencepiece transformers==5.5.4 tqdm onnx wandb

Cloning into 'VLA'...
remote: Enumerating objects: 389, done.
remote: Counting objects: 100% (389/389), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 389 (delta 260), reused 256 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (389/389), 210.64 KiB | 4.30 MiB/s, done.
Resolving deltas: 100% (260/260), done.
/content/VLA
branch 'qformer' set up to track 'origin/qformer'.
Switched to a new branch 'qformer'
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 162.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 141.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 168.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 201.6 MB/s  0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.4

In [2]:
# Install system dependencies and virtual display packages needed for headless rendering
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install pyvirtualdisplay gymnasium metaworld

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 101.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 98.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [metaworld]


In [ ]:
from google.colab import runtime, drive
drive.mount('/content/drive')

In [3]:
# Start virtual display so that OpenGL rendering works without error
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()
import os
os.environ['MUJOCO_GL']    = 'egl'
os.environ['EGL_DEVICE_ID'] = '0'
os.cpu_count()

8

In [4]:
from model.vla import TextEncoder, VisionEncoder, VLAConfig
import json
import os
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import chain

import gymnasium as gym
import numpy as np
import torch
from metaworld import policies, MT10, MT50
from tqdm import tqdm


In [5]:
class RunningStats:
    """Numerically stable, constant-memory population mean/std accumulator."""

    def __init__(self):
        self.count = 0
        self.mean = None
        self.m2 = None

    def update(self, values: np.ndarray):
        """Update using a batch whose last dimension contains features."""
        values = np.asarray(values, dtype=np.float64)
        if values.ndim < 2:
            raise ValueError(f"Expected at least 2 dimensions, got shape {values.shape}")
        values = values.reshape(-1, values.shape[-1])
        n = values.shape[0]
        if n == 0:
            return

        batch_mean = values.mean(axis=0)
        centered = values - batch_mean
        batch_m2 = np.einsum("ij,ij->j", centered, centered)

        if self.count == 0:
            self.count = n
            self.mean = batch_mean
            self.m2 = batch_m2
            return

        total = self.count + n
        delta = batch_mean - self.mean
        self.m2 += batch_m2 + delta * delta * (self.count * n / total)
        self.mean += delta * (n / total)
        self.count = total

    def mean_std(self):
        if self.count == 0:
            raise RuntimeError("Cannot calculate normalization stats: no samples were collected")
        variance = np.maximum(self.m2 / self.count, 0.0)
        return self.mean.astype(np.float32), np.sqrt(variance).astype(np.float32)


class BackgroundSaver:
    """Compress episode files in the background, with bounded queued memory."""

    def __init__(self, max_workers=2, max_pending=2):
        if max_workers < 1 or max_pending < 1:
            raise ValueError("max_workers and max_pending must both be >= 1")
        self.executor = ThreadPoolExecutor(max_workers=max_workers)
        self.max_pending = max_pending
        self.futures = []
        self.closed = False

    def save_task(self, path: str, data: dict):
        if self.closed:
            raise RuntimeError("BackgroundSaver is already closed")

        # Each pending job holds references to its episode arrays. Apply backpressure
        # so slow compression cannot build an unbounded queue of multi-GB episodes.
        while len(self.futures) >= self.max_pending:
            self._wait_oldest()

        future = self.executor.submit(self._write, path, data)
        self.futures.append(future)

    @staticmethod
    def _write(path: str, data: dict):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        tmp_path = path + ".tmp"
        t0 = time.perf_counter()
        try:
            # Open the temporary file explicitly so NumPy doesn't append '.npz'
            # to the temporary filename. os.replace makes the final publish atomic.
            with open(tmp_path, "wb") as f:
                np.savez_compressed(f, **data)
            os.replace(tmp_path, path)
            mb = os.path.getsize(path) / 1e6
            print(f"  [saver] {os.path.basename(path)} "
                  f"({mb:.1f} MB, {time.perf_counter() - t0:.1f}s)")
        except Exception:
            try:
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)
            finally:
                raise

    def _wait_oldest(self):
        future = self.futures.pop(0)
        future.result()  # Propagate save errors to the main thread.

    def wait(self):
        pending = self.futures
        self.futures = []
        errors = []
        for future in as_completed(pending):
            try:
                future.result()
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError(f"{len(errors)} background save(s) failed") from errors[0]

    def close(self):
        if self.closed:
            return
        try:
            self.wait()
        finally:
            self.executor.shutdown(wait=True)
            self.closed = True


class VLAEmbedWriter:
    """Collect one vectorized episode and write it directly in final format.

    Episode NPZ schema (matching the output of the old reformatter):
      images:        (T, num_tasks, patches, embed_dim)
      states:        (T, num_tasks, state_dim)
      actions:       (T, num_tasks, action_dim)
      chunk_indices: (T, chunk_size)
      task_names:    (num_tasks,)
    """

    def __init__(self, cfg, save_dir, task_names, saver: BackgroundSaver):
        os.makedirs(save_dir, exist_ok=True)
        self.save_dir = save_dir
        self.task_names = tuple(task_names)
        self.saver = saver
        self.episode_count = 0
        self.chnk = cfg.chunk_size
        self.action_stats = RunningStats()
        self.state_stats = RunningStats()

    def start_episode(self):
        self._images_list = []
        self._states_list = []
        self._actions_list = []
        self._chunk_list = []
        self._step = 0
        self._idxq = deque(maxlen=self.chnk)
        self._idxq.extend(np.zeros(self.chnk, dtype=np.int64))

    def add_step(self, imgs: np.ndarray, states: np.ndarray, actions: np.ndarray):
        t = self._step
        self._idxq.append(t)
        self._images_list.append(np.asarray(imgs, dtype=np.float32))
        self._states_list.append(np.asarray(states, dtype=np.float32))
        self._actions_list.append(np.asarray(actions, dtype=np.float32))
        self._chunk_list.append(np.array(self._idxq, dtype=np.int64))
        self._step += 1

    def end_episode(self, end_points = None):
        T = self._step
        if T == 0:
            return

        images = np.stack(self._images_list, axis=0)
        states = np.stack(self._states_list, axis=0)
        actions = np.stack(self._actions_list, axis=0)
        chunk_idxs = np.stack(self._chunk_list, axis=0)

        # Stats are updated only for completed/saved episodes, matching the old
        # post-hoc scan (episodes that timed out were never written to disk).
        self.state_stats.update(states)
        self.action_stats.update(actions)


        # Drop the per-step list references before handing the stacked episode to
        # the saver. The arrays remain alive while the background job needs them.
        del self._images_list, self._states_list, self._actions_list, self._chunk_list

        episode_data = {
            "images": images,
            "states": states,
            "actions": actions,
            "chunk_indices": chunk_idxs,
            "task_names": np.asarray(self.task_names, dtype=np.str_),
        }
        if end_points is not None:
            episode_data["end_points"] = end_points
        path = os.path.join(self.save_dir, f"ep{self.episode_count:04d}.npz")
        print(f"[ep {self.episode_count:04d}] {T} steps × {len(self.task_names)} tasks "
              f"→ writing one episode file: {os.path.basename(path)}")
        self.saver.save_task(path, episode_data)
        self.episode_count += 1

    def save_norm_stats(self):
        action_mean, action_std = self.action_stats.mean_std()
        state_mean, state_std = self.state_stats.mean_std()

        # Dims that are constant in the dataset (e.g. the unused 2nd-object slot):
        # identity transform, so an unexpected nonzero value at eval can't blow up.
        dead = state_std < 1e-6
        state_mean = np.where(dead, 0.0, state_mean)
        state_std = np.where(dead, 1.0, np.maximum(state_std, 1e-2))
        action_std = np.maximum(action_std, 1e-2)

        stats_path = os.path.join(self.save_dir, "norm_stats.npz")
        np.savez_compressed(
            stats_path,
            action_mean=action_mean.astype(np.float32),
            action_std=action_std.astype(np.float32),
            state_mean=state_mean.astype(np.float32),
            state_std=state_std.astype(np.float32),
        )
        print(f"Saved normalization stats to {stats_path}")
        print("dead state dims:", np.where(dead)[0])
        print("action mean:", action_mean, "action std:", action_std)
        print("state mean:", state_mean, "state std:", np.maximum(state_std, 1e-3))


class BatchAgent:
    def __init__(self, agent_classes):
        self.agents = [cls() for cls in agent_classes]

    def get_action(self, obs: np.ndarray) -> np.ndarray:
        return np.stack([agent.get_action(obs[i]) for i, agent in enumerate(self.agents)])

    def reset(self):
        for agent in self.agents:
            agent.reset()

In [6]:
cfg = VLAConfig(chunk_size=16)
def encode_text(prompt_path, device):
    with open(prompt_path, "r", encoding="utf-8") as f:
        prompts = json.load(f)
    with torch.inference_mode():
        encoder = TextEncoder(cfg).to(device)
        tokens = encoder.tokenize(list(chain.from_iterable(prompts.values())))
        embeddings, masks = encoder(tokens.to(device))
        embeddings = embeddings.detach().cpu().numpy()
        masks = masks.detach().cpu().numpy()
    return {
        prompt: (embeddings[3 * i: 3 * (i + 1)], masks[3 * i: 3 * (i + 1)])
        for i, prompt in enumerate(prompts.keys())
    }


SAVE_DIR = "/content/mt"
EPISODES = 20
MAX_STEPS = 200
SEED = 9160
MT = MT50

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# TODO: we shouldn't throw away everything just because some tasks failed.
def main():
    os.makedirs(SAVE_DIR, exist_ok=True)
    policy_names = tuple(MT().train_classes.keys())
    agent_classes = [policies.ENV_POLICY_MAP[name] for name in policy_names]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    env = gym.make_vec(
        "Meta-World/custom-mt-envs",
        vector_strategy="async",
        render_mode="rgb_array",
        seed=SEED,
        envs_list=policy_names,
        width=cfg.img_size,
        height=cfg.img_size,
        camera_name="gripperPOV",
    )
    agent = BatchAgent(agent_classes)
    saver = BackgroundSaver(max_workers=2, max_pending=2)
    writer = VLAEmbedWriter(cfg, SAVE_DIR, task_names=policy_names, saver=saver)
    encoder = VisionEncoder(cfg).to(device)
    n_eps = 0
    seed = SEED

    try:
        while n_eps < EPISODES:
            seed += 1
            obs, _info = env.reset(seed=seed)
            writer.start_episode()
            success_state = np.zeros(len(policy_names), dtype=np.uint8)
            for step in tqdm(range(MAX_STEPS)):
                actions = np.clip(agent.get_action(obs), -1, 1)
                imgs = np.asarray(env.render())[..., ::-1]  # BGR to RGB
                with torch.inference_mode():
                    imgs = torch.from_numpy(imgs.copy()).permute(0, 3, 1, 2).to(device)
                    embeds = encoder(imgs).cpu().numpy()

                obs = np.hstack([obs[..., :18], obs[..., 36:]])
                writer.add_step(embeds, obs, actions)
                obs, _reward, terminated, truncated, _info = env.step(actions)
                completion = np.logical_or(success_state, _info["success"])
                for i in range(len(policy_names)):
                    if completion[i] and not success_state[i]:
                        success_state[i] = step
                if not np.any(success_state == 0):
                    n_eps += 1
                    print(f"All tasks succeeded at {step + 1} steps")
                    print(success_state)
                    writer.end_episode(success_state)
                    break
            else:
                n_success = len(np.where(_info["success"])[0])
                print(
                    f"Exhausted all steps. {n_success} tasks failed. "
                    f"Trying another seed..."
                )

        print("Waiting for episode saves to finish...")
        saver.wait()
        writer.save_norm_stats()
        print(f"Saved text embeddings to {os.path.join(SAVE_DIR, 'text_embeds.npz')}")
    finally:
        env.close()
        saver.close()


if __name__ == "__main__":
    main()

Using device: cuda


config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/metaworld/policies/policy.py:49: UserWarning: Constant(s) may be too high. Environments clip response to [-1, 1]
  warnings.warn(
 76%|███████▋  | 153/200 [07:35<02:17,  2.92s/it]

... (186 outputs left out)

All tasks succeeded at 143 steps
[ 82  94 132 107  60  56  58  61  35  78  56  72 142  65  57  70  44  59
  77  87  60  65  62  33  29  69  89  82  82 124  51  48  46  75  47  88
  86  85  73  97  63  69  83  44  41  87  52  90  79  84]


 71%|███████   | 142/200 [06:56<02:50,  2.94s/it]

[ep 0019] 143 steps × 50 tasks → writing one episode file: ep0019.npz
Waiting for episode saves to finish...


  [saver] ep0019.npz (4001.5 MB, 197.2s)
Saved normalization stats to /content/mt/norm_stats.npz
dead state dims: [15 16 17]
action mean: [ 0.00223859  0.1580994  -0.1643401   0.3692481 ] action std: [0.47341383 0.51284456 0.6299746  0.7511802 ]
state mean: [ 0.01574518  0.67719203  0.15550727  0.5942332   0.01582632  0.6947453
  0.10204966  0.42277965 -0.04149784  0.02903439  0.44256744  0.0167432
  0.03791218  0.00748     0.02        0.          0.          0.
  0.02905051  0.7269671   0.11595528] state std: [0.1222998  0.11635947 0.09007818 0.2955894  0.12683679 0.10381869
 0.09646767 0.45979348 0.16980621 0.13339524 0.474118   0.06929074
 0.15078396 0.02971548 0.14       1.         1.         1.
 0.14831701 0.11716139 0.09803723]
Saved text embeddings to /content/mt/text_embeds.npz


In [11]:
import os, shutil
os.makedirs("/content/drive/MyDrive/VLA/preencoded/mt50_grip", exist_ok=True)
shutil.copytree("/content/mt", "/content/drive/MyDrive/VLA/preencoded/mt50_grip", dirs_exist_ok=True)
time.sleep(120)  # wait 5m for everything to finish up
runtime.unassign()

NameError: name 'runtime' is not defined